In [0]:
from pyspark.sql import functions as F

# Read unified Silver table
df = spark.table("job_market.silver.jobs")

print("Total Silver records:", df.count())

display(df.limit(10))

Total Silver records: 2221


source company ingestion_timestamp ingestion_date job_id job_title department office team job_level employment_type location location_normalized all_locations workplace_type is_remote description posted_at updated_at salary_min salary_max salary_currency salary_interval job_url apply_url lever Zoox 2026-10-06T12:52:06.606Z 2026-10-06 000392a0-3844-47c4-b580-d56d9a97620c Engineering Manager, Onboard Execution Performance Software null Software Systems null Full-time Foster City, CA foster city, ca List(Foster City, CA, Seattle, WA) hybrid false  
Zoox is building the world's most advanced self-driving hardware and software solution. The efficiency demands of such a system require an expert fine tuning of both the compute hardware architecture as well as the algorithms and middleware that runs on it to achieve maximum throughput at the most optimal power levels. 

The Software Performance team keeps the main onboard computer healthy and performant to ensure that our on-bot autonomy software meets its latency targets while also accommodating its safety goals. We are responsible for the efficient allocation of onboard compute resources (CPU/GPU/Memory/accelerators) ensuring contention is minimized and the autonomy stack executes harmoniously.

As the Onboard Execution team leader within the Performance team,  you will lead a highly specialized team of software performance engineers in aggressively optimizing on-bot software runtime. This is a high impact role that sits at the intersection of ML Platform, Core, and Autonomy Software - ensuring that efficient compute utilization is a central tenet emphasized across the organization. 
 
 
 2026-04-17T15:58:31.000Z null 237000.0 322000.0 USD per-year-salary https://jobs.lever.co/zoox/000392a0-3844-47c4-b580-d56d9a97620c https://jobs.lever.co/zoox/000392a0-3844-47c4-b580-d56d9a97620c/apply lever Zoox 2026-10-06T12:52:06.606Z 2026-10-06 00bede24-f74d-4447-ace1-48638965382e Senior / Staff Software Engineer - Motion Planning Software null Autonomy Software null Full-time Foster City, CA foster city, ca List(Foster City, CA, Boston, MA) hybrid false On the Motion Planning team at Zoox, you’ll be dedicated to improving the driving behaviors of the robot on public roads. The Planner's job is to figure out where the robot is going and how to get there safely in situations as complex as those found in Downtown San Francisco or on the Las Vegas Strip.
 
This is a cutting-edge problem, which requires teamwork and collaboration across a broad set of experts spanning Planner, Perception, Prediction, Operations, and more. Together, you will create the next generation of algorithmic improvements to expand our robot’s ability to drive in some of the most complex situations found on public roads.
 
Safety is foundational at Zoox, so you will take a measured approach – develop a principled modification to the Planner, test in simulation and on closed courses, and team up with our safety drivers for on-road testing. 
 2025-11-05T20:41:44.000Z null 254000.0 350000.0 USD per-year-salary https://jobs.lever.co/zoox/00bede24-f74d-4447-ace1-48638965382e https://jobs.lever.co/zoox/00bede24-f74d-4447-ace1-48638965382e/apply lever Palantir 2026-10-06T12:52:06.606Z 2026-10-06 00c2c97b-8514-4617-9883-e53e486b6dcd Forward Deployed Enablement Engineer - Customer Success null null Engineering null Full-time London, United Kingdom london, united kingdom List(London, United Kingdom) hybrid false A World-Changing Company
 
Palantir builds the world’s leading software for data-driven decisions and operations. By bringing the right data to the people who need it, our platforms empower our partners to develop lifesaving drugs, forecast supply chain disruptions, locate missing children, and more.


The Role
 
Forward Deployed Enablement Engineers are embedded with centralised customer success teams to maximise the outcomes of our deployed products and workflows across all commercial customers from small-scale start-ups to l

In [0]:
# ============================================================
# DATA QUALITY RULES
# ============================================================

dq_rules = F.array(

    # 1. Source validation
    F.when(
        F.col("source").isNull() |
        (~F.lower(F.trim(F.col("source"))).isin("greenhouse", "lever")),
        F.lit("INVALID_SOURCE")
    ),

    # 2. Company validation
    F.when(
        F.col("company").isNull() |
        (F.trim(F.col("company")) == ""),
        F.lit("MISSING_COMPANY")
    ),

    # 3. Job ID validation
    F.when(
        F.col("job_id").isNull() |
        (F.trim(F.col("job_id")) == ""),
        F.lit("MISSING_JOB_ID")
    ),

    # 4. Job title validation
    F.when(
        F.col("job_title").isNull() |
        (F.trim(F.col("job_title")) == ""),
        F.lit("MISSING_JOB_TITLE")
    ),

    # 5. Job URL validation
    F.when(
        F.col("job_url").isNull() |
        (F.trim(F.col("job_url")) == "") |
        (~F.col("job_url").rlike(r"^https?://")),
        F.lit("INVALID_JOB_URL")
    ),

    # 6. Posted date validation
    F.when(
        F.col("posted_at").isNull(),
        F.lit("MISSING_POSTED_AT")
    ),

    # 7. Salary minimum validation
    F.when(
        F.col("salary_min") < 0,
        F.lit("NEGATIVE_SALARY_MIN")
    ),

    # 8. Salary maximum validation
    F.when(
        F.col("salary_max") < 0,
        F.lit("NEGATIVE_SALARY_MAX")
    ),

    # 9. Salary range consistency
    F.when(
        F.col("salary_min").isNotNull() &
        F.col("salary_max").isNotNull() &
        (F.col("salary_min") > F.col("salary_max")),
        F.lit("INVALID_SALARY_RANGE")
    ),

    # 10. Workplace type validation
    F.when(
        F.col("workplace_type").isNotNull() &
        (~F.lower(F.trim(F.col("workplace_type")))
         .isin("remote", "hybrid", "onsite")),
        F.lit("INVALID_WORKPLACE_TYPE")
    )

)

In [0]:
# ============================================================
# APPLY DQ RULES
# ============================================================

dq_df = (
    df

    # Create array of possible DQ failures
    .withColumn("_dq_reasons_raw", dq_rules)

    # Remove NULL rule results
    .withColumn(
        "_dq_reasons",
        F.expr("filter(_dq_reasons_raw, x -> x IS NOT NULL)")
    )

    # PASS / FAIL
    .withColumn(
        "quality_status",
        F.when(
            F.size(F.col("_dq_reasons")) == 0,
            F.lit("PASS")
        ).otherwise(
            F.lit("FAIL")
        )
    )

    # Combine multiple failures into one readable string
    .withColumn(
        "quality_reason",
        F.when(
            F.size(F.col("_dq_reasons")) == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.array_join(F.col("_dq_reasons"), "; ")
        )
    )

    # Record when DQ was performed
    .withColumn(
        "dq_processed_at",
        F.current_timestamp()
    )

    # Remove temporary columns
    .drop("_dq_reasons_raw", "_dq_reasons")
)

display(dq_df.limit(20))

source company ingestion_timestamp ingestion_date job_id job_title department office team job_level employment_type location location_normalized all_locations workplace_type is_remote description posted_at updated_at salary_min salary_max salary_currency salary_interval job_url apply_url quality_status quality_reason dq_processed_at greenhouse Cloudflare 2026-10-06T12:44:37.740Z 2026-10-06 5390243 Senior Software Engineer, Durable Objects (DO) Emerging Technology and Incubation Amsterdam, Netherlands null null null In-Office in-office List(In-Office) null false <div class="content-intro"><h3>About Us</h3>
<p>At Cloudflare, we are on a mission to help build a better Internet. Today the company runs one of the world’s largest networks that powers millions of websites and other Internet properties for customers ranging from individual bloggers to SMBs to Fortune 500 companies. Cloudflare protects and accelerates any Internet application online without adding hardware, installing software, or changing a line of code. Internet properties powered by Cloudflare all have web traffic routed through its intelligent global network, which gets smarter with every request. As a result, they see significant improvement in performance and a decrease in spam and other attacks. Cloudflare was named to Entrepreneur Magazine’s Top Company Cultures list and ranked among the World’s Most Innovative Companies by Fast Company.</p>
<p>At Cloudflare, we’re not looking for people who wait for a polished roadmap; we’re looking for the builders who see the cracks in the Internet that everyone else has simply learned to live with. We value candidates who have the instinct to spot a "normalized" problem and the AI-native curiosity to create a solution using the latest tools. Our culture is built on iteration, leveraging AI to ship faster today to make it better tomorrow, while ensuring that every improvement, no matter how small, is shared across the team to lift everyone up. If you’re the type of person who values curiosity over bureaucracy, and that AI is a partner in solving tough problems to keep the Internet moving forward, you’ll fit right in.</p></div><h3><strong>Available Locations: Austin, TX</strong></h3>
<h3>About the Role</h3>
<p>Cloudflare’s Emerging Technologies &amp; Incubation team builds and launches bold new products on Cloudflare’s edge network. This role is on the Durable Objects team, which powers stateful serverless applications for Cloudflare customers. You’ll help evolve the runtime and the low-level routing and storage systems that support real-time chat, multiplayer games, AI agents, and other stateful workloads at scale.</p>
<h3>Responsibilities</h3>
<ul>
<li>Build and improve the Durable Objects runtime.</li>
<li>Design and develop the low-level routing and storage systems that power customer applications.</li>
<li>Improve reliability, scalability, performance, and feature depth.</li>
<li>Own code end-to-end from design through production.</li>
<li>Solve complex problems in production distributed systems.</li>
<li>Work across JavaScript, TypeScript, Rust, and C++.</li>
<li>Help drive the continued evolution of the system.</li>
</ul>
<h3>Desirable Skills, Knowledge &amp; Experience</h3>
<ul>
<li>5+ years of experience writing systems software.</li>
<li>Operational mindset with experience owning code in production.</li>
<li>Strong foundation in data structures, algorithms, and system design.</li>
<li>Deep expertise in at least one systems programming language such as C/C++ or Rust.</li>
<li>Experience designing and architecting large-scale distributed systems.</li>
<li>Familiarity with cloud platforms, especially serverless architectures.</li>
<li>Experience as a technical leader or team lead, driving engineering decisions.</li>
</ul>
<h3>Bonus Points</h3>
<ul>
<li>Experience working on database internals, especially SQLite.</li>
<li>Deep understanding of web technologies such as browsers, HTTP, JavaScript, and WebAssembly.</li>
<l

In [0]:
# ============================================================
# DQ SUMMARY
# ============================================================

dq_summary = (
    dq_df
    .groupBy("quality_status")
    .count()
    .orderBy("quality_status")
)

display(dq_summary)

quality_status,count
PASS,2221


In [0]:
# QUALITY PERCENTAGE


total_records = dq_df.count()

passed_records = (
    dq_df
    .filter(F.col("quality_status") == "PASS")
    .count()
)

failed_records = (
    dq_df
    .filter(F.col("quality_status") == "FAIL")
    .count()
)

pass_percentage = (
    (passed_records / total_records) * 100
    if total_records > 0
    else 0
)

print("======================================")
print("DATA QUALITY SUMMARY")
print("======================================")
print(f"Total records : {total_records}")
print(f"Passed        : {passed_records}")
print(f"Failed        : {failed_records}")
print(f"Pass %        : {pass_percentage:.2f}%")
print("======================================")

DATA QUALITY SUMMARY
Total records : 2221
Passed        : 2221
Failed        : 0
Pass %        : 100.00%


In [0]:
# DQ FAILURE ANALYSIS


dq_failures = (
    dq_df
    .filter(F.col("quality_status") == "FAIL")
    .withColumn(
        "dq_reason",
        F.explode(F.split(F.col("quality_reason"), "; "))
    )
    .groupBy("dq_reason")
    .count()
    .orderBy(F.desc("count"))
)

display(dq_failures)

dq_reason,count


In [0]:
# ============================================================
# VALID RECORDS
# ============================================================

valid_jobs = (
    dq_df
    .filter(F.col("quality_status") == "PASS")
)

# ============================================================
# INVALID RECORDS
# ============================================================

quarantined_jobs = (
    dq_df
    .filter(F.col("quality_status") == "FAIL")
)

print("Valid records      :", valid_jobs.count())
print("Quarantined records:", quarantined_jobs.count())

Valid records      : 2221
Quarantined records: 0


In [0]:
# ============================================================
# WRITE VALIDATED JOBS
# ============================================================

valid_jobs.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("job_market.silver.jobs_validated")

print("Validated Silver table created:")
print("job_market.silver.jobs_validated")

Validated Silver table created:
job_market.silver.jobs_validated


In [0]:
# ============================================================
# QUARANTINE LOCATION
# ============================================================

quarantine_path = (
    "s3://job-market-analysis-pipeline/"
    "quarantine-data/jobs/"
)

# ============================================================
# WRITE FAILED RECORDS
# ============================================================

quarantined_jobs.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("ingestion_date") \
    .save(quarantine_path)

print("Quarantine data written to:")
print(quarantine_path)

Quarantine data written to:
s3://job-market-analysis-pipeline/quarantine-data/jobs/


In [0]:
# ============================================================
# REGISTER QUARANTINE DELTA TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS job_market.silver.quarantined_jobs
USING DELTA
LOCATION '{quarantine_path}'
""")

print("Quarantine table registered:")
print("job_market.silver.quarantined_jobs")

Quarantine table registered:
job_market.silver.quarantined_jobs


In [0]:
# ============================================================
# REGISTER QUARANTINE DELTA TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS job_market.silver.quarantined_jobs
USING DELTA
LOCATION '{quarantine_path}'
""")

print("Quarantine table registered:")
print("job_market.silver.quarantined_jobs")

Quarantine table registered:
job_market.silver.quarantined_jobs


In [0]:
# ============================================================
# DQ RUN SUMMARY
# ============================================================

dq_run_summary = spark.createDataFrame(
    [
        (
            total_records,
            passed_records,
            failed_records,
            float(pass_percentage)
        )
    ],
    [
        "total_records",
        "passed_records",
        "failed_records",
        "pass_percentage"
    ]
).withColumn(
    "dq_run_timestamp",
    F.current_timestamp()
)

display(dq_run_summary)

total_records,passed_records,failed_records,pass_percentage,dq_run_timestamp
2221,2221,0,100.0,2026-10-06T16:06:48.813Z


In [0]:
# ============================================================
# SAVE DQ MONITORING RESULT
# ============================================================

dq_run_summary.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("job_market.monitoring.data_quality_summary")

print("DQ monitoring summary saved.")

DQ monitoring summary saved.


In [0]:
print(
    "Original Silver:",
    spark.table("job_market.silver.jobs").count()
)

Original Silver: 2221


In [0]:
print(
    "Validated:",
    spark.table("job_market.silver.jobs_validated").count()
)

Validated: 2221


In [0]:
print(
    "Quarantined:",
    spark.table("job_market.silver.quarantined_jobs").count()
)

Quarantined: 0


In [0]:
original_count = spark.table(
    "job_market.silver.jobs"
).count()

validated_count = spark.table(
    "job_market.silver.jobs_validated"
).count()

quarantine_count = spark.table(
    "job_market.silver.quarantined_jobs"
).count()

print("======================================")
print("DQ RECONCILIATION")
print("======================================")
print("Original Silver :", original_count)
print("Validated       :", validated_count)
print("Quarantined     :", quarantine_count)
print("Sum             :", validated_count + quarantine_count)
print("Difference      :", original_count - (validated_count + quarantine_count))
print("======================================")

DQ RECONCILIATION
Original Silver : 2221
Validated       : 2221
Quarantined     : 0
Sum             : 2221
Difference      : 0
